# Figure: Saturation Pressures

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

from helpers.plot_styles import (
    AXIS_LABEL_FONTSIZE,
    LEGEND_STYLE,
    PANEL_TITLE_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    SAVE_DPI,
    TICK_FONTSIZE,
    TOOL_MARKER_STYLE_w_HC_MODEL
)

satp_df = pd.read_csv(results_directory / "saturation_pressures.csv")
samples = ["MORB", "Kilauea", "Fuego", "Fogo"]
satp_df = satp_df.set_index("Sample").loc[samples]

## Compute stats

In [ ]:
# compute mean, median, and standard deviation for plotting
satp_cols = [c for c in satp_df.columns if c.endswith('_SatP_bars')]
satp_df['median'] = satp_df[satp_cols].median(axis=1)
satp_df['min'] = satp_df[satp_cols].min(axis=1)
satp_df['max'] = satp_df[satp_cols].max(axis=1)

## Build the figure

Each sample gets its own vertical panel. Within a panel, core tools (DCompress, EVo, MAGEC, SulfurX, VolFe) are laid out first; a small gap separates the VESIcal implemented tools (IM, VC, MS) clustered to the right. Sample names are centered at the top of each panel, and a single legend at the lower-center of the axes covers every tool.

In [ ]:
# --- Choose tools and samples to plot --- #
tools  = [c.replace("_SatP_bars", "") for c in satp_df.columns if c.endswith("_SatP_bars")]

core_tools    = [m for m in tools if not m.startswith("VESIcal")]
vesical_tools = [m for m in tools if m.startswith("VESIcal")]

# --- Panel layout: x-offsets within each panel --- #
CORE_SPACING    = 2.5     # x units between consecutive core tool markers
VESICAL_SPACING = 1     # x units between VESIcal sub-tools
GROUP_GAP       = 3     # extra gap between core and VESIcal groups
PANEL_PADDING   = 4   # x units between panel divider and nearest marker

offsets = {}
x = 0
for m in core_tools:
    offsets[m] = x
    x += CORE_SPACING
if vesical_tools:
    x += GROUP_GAP
    for m in vesical_tools:
        offsets[m] = x
        x += VESICAL_SPACING

last_offset   = max(offsets.values())                    # x of rightmost marker (per panel)
panel_width   = last_offset + 2 * PANEL_PADDING          # identical for every panel
panel_bases   = {s: i * panel_width for i, s in enumerate(samples)}
panel_centers = [panel_bases[s] + last_offset / 2 for s in samples]
dividers      = [(i + 1) * panel_width - PANEL_PADDING for i in range(len(samples) - 1)]

# --- Draw the figure --- #
fig, ax = plt.subplots(figsize=(11.5, 5))

for tool in tools:
    style = TOOL_MARKER_STYLE_w_HC_MODEL.get(tool)
    if style is None:
        continue
    label, marker, size, facecolor, edgecolor, lw = style
    col = f"{tool}_SatP_bars"

    xs, ys = [], []
    for index, row in satp_df.iterrows():
        val = row[col]
        if pd.notna(val):
            xs.append(panel_bases[index] + offsets[tool])
            ys.append(val)

    # Line-only markers (x, +, |, _) don't accept facecolors
    if marker in ("x", "+", "|", "_"):
        ax.scatter(xs, ys, marker=marker, s=size, color=edgecolor,
                   linewidths=lw, label=label, zorder=3)
    else:
        ax.scatter(xs, ys, marker=marker, s=size,
                   facecolors=facecolor, edgecolors=edgecolor,
                   linewidths=lw, label=label, zorder=3)

# Vertical dividers between sample panels
for xd in dividers:
    ax.axvline(xd, color="k", linewidth=0.8, zorder=1)

ax.set_xlim(-PANEL_PADDING, len(samples) * panel_width - PANEL_PADDING)
ax.set_ylim(0, 8000)

# Sample names centered at top of each panel
y_top = ax.get_ylim()[1]
for cx, sample in zip(panel_centers, samples):
    ax.text(cx, y_top * 0.95, SAMPLE_DISPLAY_NAMES.get(sample, sample),
            ha="center", va="top", fontsize=PANEL_TITLE_FONTSIZE)

ax.set_ylabel("Saturation Pressure (bar)", fontsize=AXIS_LABEL_FONTSIZE)
ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_xticks([])

# add mean, median, range
box_x_lims = [(-10, 24), (24.1, 52), (52.1, 80), (80.1, 120)]

for lims, sample in zip(box_x_lims, satp_df.index):
    ax.fill_between(x=lims,
                    y1=satp_df["min"][sample],
                    y2=satp_df["max"][sample],
                    alpha=0.1,
                    color="gray")
    ax.hlines(satp_df["median"][sample], xmin=lims[0], xmax=lims[1],
              color="#A6A6A6", linestyle="--")

ax.legend(**LEGEND_STYLE, bbox_to_anchor=(0.51, 0.2), ncol=2)
fig.set_layout_engine("constrained")

if SAVE_FIG:
    fig.savefig("figures/Fig_saturation_pressures.png", dpi=SAVE_DPI, bbox_inches="tight")

plt.show()

## Compute densities and depths

In [ ]:
# --- Caclulate equivalent depth --- #
def pressure_to_depth(pressure_bar, rho_shallow, rho_deep, transition_depth_km):
    """Converts pressure in bars to depth in km on the Earth. Allows multiple
    crustal density values to calculate pressure over a depth range that encompasses
    two zones with distinct densities. For example, MORB depth below sea level combines
    1 g/cm3 for seawater with 2.5 g/cm3 for basaltic crust.

    Parameters
    ----------
    pressure_bar (float)
        Pressure in bars
    rho_shallow (float)
        Density of a shallow region in g/cm^3
    rho_deep (float)
        Density of a deep region in g/cm^3
    transition_depth_km (float)
        Depth of transition from shallow to deep substrate type in km
    
    Returns
    -------
    float
        Depth in km
    """
    g        = 9.807
    P_Pa     = pressure_bar * 1e5
    rho_shallow_km_m3 = rho_shallow * 1000
    rho_deep_km_m3 = rho_deep * 1000
    transition_depth_m = transition_depth_km * 1000
    
    pressure_shallow_Pa  = rho_shallow_km_m3 * g * transition_depth_m   # Pa
    pressure_deep_Pa     = P_Pa - pressure_shallow_Pa
    thickness_deepzone_m       = pressure_deep_Pa / (rho_deep_km_m3 * g)

    return (transition_depth_m + thickness_deepzone_m) / 1000          # m -> km

crust_profiles = {
    "MORB": {
        "crustal_density_shallow": 1.0, # seawater
        "crustal_density_deep": 2.9,
        "depth_to_transition_km": 2.75,
        "density_reference": "Theunissen et al. (2022)",
    },
    "Kilauea": {
        "crustal_density_shallow": 2.3,
        "crustal_density_deep": 2.3,
        "depth_to_transition_km": 0,
        "density_reference": "Denlinger and Flinders (2022)",
    },
    "Fuego": {
        "crustal_density_shallow": 2.38,
        "crustal_density_deep": 2.8,
        "depth_to_transition_km": 6,
        "density_reference": "Mickus (2003)",
    },
    "Fogo": {
        "crustal_density_shallow": 2.7,
        "crustal_density_deep": 3.1,
        "depth_to_transition_km": 12,
        "density_reference": "Maria Lo Forte et al. (2023)",
    },
}

## Write the saturation-pressure + depth table

In [ ]:
from IPython.display import display

TOOL_DISPLAY = {
    "DCompress":              "D-Compress",
    "DCompress (IM)":         "D-Compress (IM)",
    "EVo":                    "EVo",
    "MAGEC":                  "MAGEC",
    "SulfurX":                r"Sulfur\_X",
    "VolFe":                  "VolFe",
    "VESIcal_Iacono":         "VESIcal (IM)",
    "VESIcal_IaconoMarziano": "VESIcal (IM)",
    "VESIcal_Dixon":          "VESIcal (VC)",
    "VESIcal_MS":             "VESIcal (MS)",
}
TOOL_ORDER = list(dict.fromkeys(TOOL_DISPLAY.keys()))
SAMPLE_DISPLAY_LATEX = {"Kilauea": r"K\={\i}lauea"}


def _fmt_p(v):
    return "---" if pd.isna(v) else f"{round(float(v)):d}"

def _fmt_d(v):
    return "---" if pd.isna(v) else f"{float(v):.1f}"

tools_in_df = [c[:-len("_SatP_bars")] for c in satp_df.columns if c.endswith("_SatP_bars")]
table_tools = [t for t in TOOL_ORDER if t in tools_in_df] + [t for t in tools_in_df if t not in TOOL_ORDER]

# Compute pressure + depth
def _depth(p, s):
    if pd.isna(p):
        return float("nan")
    profile = crust_profiles[s]
    return pressure_to_depth(
        p, profile["crustal_density_shallow"], profile["crustal_density_deep"], profile["depth_to_transition_km"]
    )

# One (P, depth) cell-pair per sample for a given per-sample pressure lookup
def _row_cells(get_p):
    drow, lrow = {}, []
    for s in samples:
        p = get_p(s)
        d = _depth(p, s)
        sd = SAMPLE_DISPLAY_NAMES.get(s, s)
        drow[(sd, "P (bars)")] = p
        drow[(sd, "Depth (km)")] = d
        lrow += [_fmt_p(p), _fmt_d(d)]
    return drow, lrow

display_rows, latex_rows = {}, {}
for tool in table_tools:
    col = f"{tool}_SatP_bars"
    drow, lrow = _row_cells(lambda s, col=col: satp_df.loc[s, col])
    display_rows[TOOL_DISPLAY.get(tool, tool).replace(r"\_", "_")] = drow
    latex_rows[tool] = lrow

# Summary rows: blank spacer, then per-sample min/median/max across all tools
summary_rows = {}
for label, key in (("Minimum", "min"), ("Median", "median"), ("Maximum", "max")):
    drow, lrow = _row_cells(lambda s, key=key: satp_df.loc[s, key])
    display_rows[label] = drow
    summary_rows[label] = lrow

# --- Pretty-printed table in the notebook --- #
table_display = pd.DataFrame.from_dict(display_rows, orient="index")
table_display.columns = pd.MultiIndex.from_tuples(table_display.columns)
table_display.index.name = "Tool"

fmt_map = {}
for s in samples:
    sd = SAMPLE_DISPLAY_NAMES.get(s, s)
    fmt_map[(sd, "P (bars)")] = lambda v: "---" if pd.isna(v) else f"{v:,.0f}"
    fmt_map[(sd, "Depth (km)")] = lambda v: "---" if pd.isna(v) else f"{v:.1f}"

# Put a little space + a rule above the Minimum/Median/Maximum summary block
def _summary_sep(row):
    border = "border-top: 1px solid black;" if row.name == "Minimum" else ""
    pad = "padding-top: 0.6em;" if row.name == "Minimum" else ""
    return [border + pad] * len(row)

display(
    table_display.style
    .format(fmt_map)
    .apply(_summary_sep, axis=1)
    .set_caption("Vapor saturation pressure and equivalent depth by tool")
)

# --- LaTeX output written to file --- #
n = len(samples)
col_spec = "l" + "rr" * n
top_cells = [r"\multirow{2}{*}{Tool}"] + [
    rf"\multicolumn{{2}}{{c}}{{{SAMPLE_DISPLAY_LATEX.get(s, s)}}}" for s in samples
]
clines = " ".join(rf"\cline{{{2 + 2 * i}-{3 + 2 * i}}}" for i in range(n))
sub_cells = [""] + [c for _ in samples for c in ("bars", "km")]

lines = [
    r"\begin{table}",
    r"    \centering",
    rf"    \begin{{tabular}}{{{col_spec}}}",
    r"    \hline",
    "        " + " & ".join(top_cells) + r" \\",
    "        " + clines,
    "        " + " & ".join(sub_cells) + r" \\ \hline",
]
for tool in table_tools:
    lines.append("        " + " & ".join([TOOL_DISPLAY.get(tool, tool)] + latex_rows[tool]) + r" \\")
# blank vertical space below the tool block, then Minimum/Median/Maximum summary rows
lines.append(r"        \noalign{\medskip}")
for label in ("Minimum", "Median", "Maximum"):
    lines.append("        " + " & ".join([label] + summary_rows[label]) + r" \\")
lines += [
    r"    \hline",
    r"    \end{tabular}",
    r"    \caption{Vapor saturation pressure ($P_\mathrm{v}^\mathrm{sat}$, in bars) "
    r"and equivalent depth (km) calculated by each tool for each basalt. Depths are "
    r"derived from each tool's saturation pressure using the crustal density profiles "
    r"described in the text.}",
    r"    \label{table:SatP_depth}",
    r"\end{table}",
    "",
]
table_str = "\n".join(lines)


In [ ]:
# --- Preview the LaTeX table as it will actually compile --- #
import subprocess, tempfile, pathlib
from IPython.display import Image, display

def preview_latex(body, packages=("multirow",), dpi=200):
    """Compile a LaTeX snippet to a tightly-cropped PNG and display it inline.

    Falls back to printing the raw LaTeX if pdflatex/pdftoppm aren't available
    or compilation fails.
    """
    pkgs = "\n".join(rf"\usepackage{{{p}}}" for p in packages)
    doc = (
        r"\documentclass{article}" "\n"
        f"{pkgs}\n"
        r"\usepackage[active,tightpage,floats]{preview}" "\n"
        r"\begin{document}" "\n"
        f"{body}\n"
        r"\end{document}" "\n"
    )
    with tempfile.TemporaryDirectory() as d:
        d = pathlib.Path(d)
        (d / "tbl.tex").write_text(doc)
        run = subprocess.run(
            ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "tbl.tex"],
            cwd=d, capture_output=True, text=True,
        )
        if not (d / "tbl.pdf").exists():
            raise RuntimeError("pdflatex failed:\n" + run.stdout[-2000:])
        subprocess.run(["pdftoppm", "-png", "-r", str(dpi), "tbl.pdf", "tbl"], cwd=d, check=True)
        display(Image(filename=str(sorted(d.glob("tbl*.png"))[0])))

preview_latex(table_str)